# 第十一周作业：完整 SQL Agent Workflow
参考课程 sql-agent.ipynb，使用原始 chinook.db。以下执行输出来自固定离线测试替身，不是真实模型调用。自由问答切换 HTTPModel；详见 README。

In [1]:
from sql_agent import DBParser, SQLWorkflow, DemoModel, HTTPModel, CASES
import json
parser = DBParser()
print("业务表数量:", len(parser.schema))

业务表数量: 11


## 1. 数据库解析
读取字段、主外键和行数，排除系统表，不发送客户样本。

In [2]:
for name, meta in parser.schema.items():
    print(name, "行数:", meta["row_count"], "外键:", meta["foreign_keys"])

albums 行数: 347 外键: [{'from': 'ArtistId', 'table': 'artists', 'to': 'ArtistId'}]
artists 行数: 275 外键: []
customers 行数: 59 外键: [{'from': 'SupportRepId', 'table': 'employees', 'to': 'EmployeeId'}]
employees 行数: 8 外键: [{'from': 'ReportsTo', 'table': 'employees', 'to': 'EmployeeId'}]
genres 行数: 25 外键: []
invoice_items 行数: 2240 外键: [{'from': 'TrackId', 'table': 'tracks', 'to': 'TrackId'}, {'from': 'InvoiceId', 'table': 'invoices', 'to': 'InvoiceId'}]
invoices 行数: 412 外键: [{'from': 'CustomerId', 'table': 'customers', 'to': 'CustomerId'}]
media_types 行数: 5 外键: []
playlist_track 行数: 8715 外键: [{'from': 'TrackId', 'table': 'tracks', 'to': 'TrackId'}, {'from': 'PlaylistId', 'table': 'playlists', 'to': 'PlaylistId'}]
playlists 行数: 18 外键: []
tracks 行数: 3503 外键: [{'from': 'MediaTypeId', 'table': 'media_types', 'to': 'MediaTypeId'}, {'from': 'GenreId', 'table': 'genres', 'to': 'GenreId'}, {'from': 'AlbumId', 'table': 'albums', 'to': 'AlbumId'}]


## 2–6. 选表、生成、校验、改写、执行
显式注入错误字段 TotalAmount，观察先失败、修复、重新校验后执行。

In [3]:
workflow = SQLWorkflow(parser, DemoModel(inject_error=True))
result = workflow.run(CASES[0]["question"])
assert result["status"] == "success"
for event in result["trace"]:
    if event["stage"] != "database_parse":
        print(json.dumps(event, ensure_ascii=False, indent=2))

{
  "stage": "table_select",
  "tables": [
    "invoices"
  ],
  "reason": "离线固定测试用例选表"
}
{
  "stage": "sql_generate",
  "sql": "SELECT BillingCountry AS country, ROUND(SUM(TotalAmount),2) AS revenue FROM invoices GROUP BY BillingCountry ORDER BY revenue DESC, country LIMIT 5"
}
{
  "stage": "sql_validate",
  "attempt": 0,
  "ok": false,
  "error": "no such column: TotalAmount"
}
{
  "stage": "sql_rewrite",
  "attempt": 1,
  "sql": "SELECT BillingCountry AS country, ROUND(SUM(Total),2) AS revenue FROM invoices GROUP BY BillingCountry ORDER BY revenue DESC, country LIMIT 5",
  "cause": "no such column: TotalAmount"
}
{
  "stage": "sql_validate",
  "attempt": 1,
  "ok": true,
  "plan": {
    "columns": [
      "id",
      "parent",
      "notused",
      "detail"
    ],
    "rows": [
      [
        8,
        0,
        0,
        "SCAN invoices"
      ],
      [
        10,
        0,
        0,
        "USE TEMP B-TREE FOR GROUP BY"
      ],
      [
        53,
        0,
        0,
 

## 7. 结果汇总
确定性汇总只引用实际查询值；完整结果保留在 result 中。

In [4]:
print(result["summary"])
print("最终 SQL:", result["sql"])

返回 5 行。前 5 行：country=USA, revenue=523.06；country=Canada, revenue=303.96；country=France, revenue=195.1；country=Brazil, revenue=190.1；country=Germany, revenue=156.48。
最终 SQL: SELECT BillingCountry AS country, ROUND(SUM(Total),2) AS revenue FROM invoices GROUP BY BillingCountry ORDER BY revenue DESC, country LIMIT 5


## 多表、趋势、计数与空结果

In [5]:
for case in CASES[1:]:
    answer = SQLWorkflow(parser, DemoModel()).run(case["question"])
    assert answer["status"] == "success"
    print(case["question"])
    print(answer["sql"])
    print(answer["summary"])

销售额最高的五种音乐曲风是什么？
SELECT g.Name AS genre, ROUND(SUM(i.UnitPrice*i.Quantity),2) AS revenue FROM invoice_items i JOIN tracks t ON i.TrackId=t.TrackId JOIN genres g ON t.GenreId=g.GenreId GROUP BY g.GenreId,g.Name ORDER BY revenue DESC, genre LIMIT 5
返回 5 行。前 5 行：genre=Rock, revenue=826.65；genre=Latin, revenue=382.14；genre=Metal, revenue=261.36；genre=Alternative & Punk, revenue=241.56；genre=TV Shows, revenue=93.53。
2013年每月销售额是多少？
SELECT strftime('%Y-%m',InvoiceDate) AS month, ROUND(SUM(Total),2) AS revenue FROM invoices WHERE InvoiceDate >= '2013-01-01' AND InvoiceDate < '2014-01-01' GROUP BY month ORDER BY month
返回 12 行。前 5 行：month=2013-01, revenue=37.62；month=2013-02, revenue=27.72；month=2013-03, revenue=37.62；month=2013-04, revenue=33.66；month=2013-05, revenue=37.62。
共有多少位客户？
SELECT COUNT(*) AS customer_count FROM customers
返回 1 行。前 1 行：customer_count=59。
2099年有哪些发票？
SELECT InvoiceId,Total FROM invoices WHERE InvoiceDate >= '2099-01-01' AND InvoiceDate < '2100-01-01' ORDER BY InvoiceId


## 安全与失败分支

In [6]:
print(parser.check_sql("DELETE FROM customers", ["customers"]))
print(parser.check_sql("SELECT * FROM employees", ["customers"]))
print(SQLWorkflow(parser, DemoModel()).run("不在演示列表中的问题")["status"])

{'ok': False, 'error': '仅允许 SELECT 或只读 WITH 查询'}
{'ok': False, 'error': 'access to employees.EmployeeId is prohibited'}
failed


## 自动化验证

In [7]:
import unittest
import test_sql_agent
suite = unittest.defaultTestLoader.loadTestsFromModule(test_sql_agent)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()

test_invalid_selection (test_sql_agent.WorkflowTests) ... ok
test_known_results (test_sql_agent.WorkflowTests) ... ok
test_readonly_and_selection (test_sql_agent.WorkflowTests) ... ok
test_retry_bound (test_sql_agent.WorkflowTests) ... ok
test_rewrite (test_sql_agent.WorkflowTests) ... ok
test_schema (test_sql_agent.WorkflowTests) ... ok
test_timeout (test_sql_agent.WorkflowTests) ... ok
test_truncation (test_sql_agent.WorkflowTests) ... ok
test_unknown_demo_question (test_sql_agent.WorkflowTests) ... ok

----------------------------------------------------------------------
Ran 9 tests in 0.083s

OK


## 真实模型接入（未执行）
在环境变量中设置 CHATBI_API_KEY、CHATBI_BASE_URL、CHATBI_MODEL 后，运行下列代码。不要将密钥填进 notebook。
```python
agent = SQLWorkflow(parser, HTTPModel())
answer = agent.run("各国家销售额排名前五的是哪些？")
print(answer)
```
真实模型结果仍需人工核对指标口径。人工编辑审核界面、多轮追问和图表属于设计文档中的后续范围。